<a href="https://colab.research.google.com/github/Ddkaba/IAD_Curs/blob/main/IAD_Cursach.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import SelectKBest, f_classif

In [ ]:
dataset = pd.read_csv(
    "https://raw.githubusercontent.com/Ddkaba/IAD_Curs/main/dataset.csv")
dataset.columns = [c.split(",")[0] if "," in str(c) else c for c in dataset.columns]
if "No" in dataset.columns:
    dataset = dataset.drop(columns=["No"])

In [ ]:
TARGET = dataset.columns[-1]

print("Общая информация")
print(dataset.info())

print(f"Количество записей (объектов): {dataset.shape[0]}")
print(f"Количество признаков (фич): {dataset.shape[1]}")

print("\nНазвания столбцов:")
print(dataset.columns.tolist())

print("\nТипы данных:")
print(dataset.dtypes)

print("\nПропущенные значения:")
missing_values = dataset.isnull().sum()
print(missing_values)
print(f"Общее количество пропущенных значений: {missing_values.sum()}")

print("Целевая переменная")
if TARGET in dataset.columns:
    print(f"\nЦелевая переменная: {TARGET}")
    print(f"Тип данных целевой переменной: {dataset[TARGET].dtype}")
    unique_values = dataset[TARGET].unique()
    print(f"Уникальные значения целевой переменной (первые 20): {unique_values[:20]}")
    print(f"Всего уникальных значений: {unique_values.size}")
    if dataset[TARGET].nunique() <= 20:
        print("Распределение классов:")
        print(dataset[TARGET].value_counts())
        print("Процентное соотношение классов:")
        print(dataset[TARGET].value_counts(normalize=True) * 100)

print("Статистика")
print(dataset.describe())

In [ ]:
feat = [c for c in dataset.columns if c != TARGET]
n = len(feat)
k = 4
size = (n + k - 1) // k
chunks = [feat[i : i + size] for i in range(0, n, size)]

for i, cols in enumerate(chunks):
    block = cols + [TARGET]
    corr = dataset[block].corr(numeric_only=True)
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, annot_kws={"size": 7})
    plt.title("Корреляционная матрица (признаки {}-{}, + {})".format(i * size + 1, min((i + 1) * size, n), TARGET))
    plt.tight_layout()
    plt.show()

In [ ]:
corr_with_target = dataset[feat + [TARGET]].corr(numeric_only=True)[TARGET].drop(TARGET, errors="ignore")
top2 = corr_with_target.abs().nlargest(2).index.tolist()
x_col, y_col = top2[0], top2[1]

plt.figure(figsize=(10, 8))
sns.scatterplot(data=dataset, x=x_col, y=y_col, hue=TARGET, alpha=0.7)
plt.title("Диаграмма рассеивания: {} vs {} (цвет = {})".format(x_col, y_col, TARGET))
plt.tight_layout()
plt.show()

In [ ]:
feature_columns = dataset.select_dtypes(include=[np.number]).columns.tolist()
if TARGET in feature_columns:
    feature_columns.remove(TARGET)

_ = dataset[feature_columns].hist(
    bins=10,
    figsize=(20, 15),
    grid=False,
    edgecolor="black",
)
plt.suptitle("Распределение числовых признаков", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
source_df = dataset
X_num = source_df.drop(columns=[TARGET]).select_dtypes(include=[np.number])
y = source_df[TARGET]

all_selector = SelectKBest(score_func=f_classif, k="all")
all_selector.fit(X_num, y)

scores_df = (
    pd.DataFrame({"feature": X_num.columns, "score": all_selector.scores_})
    .sort_values("score", ascending=False)
    .reset_index(drop=True)
)

print("Оценки информативности (f_classif), по убыванию:")
print(scores_df)

plt.figure(figsize=(10, 12))
sns.barplot(data=scores_df, x="score", y="feature", color="#1f77b4")
plt.title("SelectKBest: f_classif scores")
plt.tight_layout()
plt.show()

K = 5
selector = SelectKBest(score_func=f_classif, k=min(K, X_num.shape[1]))
selector.fit(X_num, y)
selected_features = X_num.columns[selector.get_support()].tolist()
print(f"Топ-{K} признаков:")
print(selected_features)